# tinybamdp tutorial — exact Bayes-adaptive models of exploration

**How to use `tinybamdp` (version 0), a small from-scratch library for tabular decision-making under uncertainty.**

When people choose between uncertain options they must trade off *exploiting* what they know against *exploring* to learn more.
Normative accounts treat this as planning in **belief space**: a Bayes-adaptive MDP (BAMDP) when the dynamics are unknown,
and a POMDP when the state is hidden. This notebook builds both up step by step and shows how to wrap them as cognitive
models of choice, simulate agents and fit them to data. All data in this notebook are simulated; no external dataset is used.

The research study built on the library (when can myopic and Bayes-optimal exploration be told apart?) is in the companion
notebook **`kg_vs_optimal_identifiability.ipynb`**.

| § | Section | What you can do |
|---|---|---|
| 0 | Setup | load the library from the repository |
| – | Quick start | simulate one subject and compare models in five lines |
| 1 | Tests | run the unit tests |
| 2 | Beliefs | update a Beta belief with buttons |
| 3 | Exact BAMDP planning | inspect Bayes-optimal vs KG values in any hyperstate |
| 4 | Value of information | Bayes-adaptive vs certainty-equivalent planning on a generic BAMDP |
| 5 | POMDP: the tiger problem | belief-space value functions and simulated episodes |
| 6 | Cognitive models | simulate five agents (2–4 arms in the widgets) and compare behavioural signatures |
| 7 | Model fitting | fit all models to one simulated subject |

**How each section is organised.** Every topic section has the same three parts:
1. **Observation**: what the cell lets you see or do, and the question it answers.
2. **Mathematical formulation**: the equations the code implements.
3. **Analysis**: what the output shows with the default settings, and how to read it.

*Every interactive panel has buttons; long outputs are in scrollable boxes. Runs on a standard Colab CPU runtime.*

## 0 · Setup
Run this cell first. It finds the repository and makes `tinybamdp` (the library) and `study` importable, without copying any code:

1. **Local Jupyter**: open the notebook from the repository (top level or `tinybamdp/notebook/`); nothing else to do.
2. **Colab** (default): the cell clones [bunytwirls/tinybamdp-library](https://github.com/bunytwirls/tinybamdp-library) automatically.
3. **Offline / no GitHub access**: set `REPO_URL = ""` and upload `tinybamdp.zip` (the repository) when asked.

In [ ]:
REPO_URL = "https://github.com/bunytwirls/tinybamdp-library.git"   # set to "" to use tinybamdp.zip instead
import os, sys, pathlib, subprocess, zipfile
try:                   # Colab: enable interactive widgets (no-op elsewhere)
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass
def _project(cand):
    # the project folder holds the tinybamdp package and study/; in the repository it is the tinybamdp/ subfolder
    for p in (cand, cand / "tinybamdp"):
        if (p / "tinybamdp" / "__init__.py").exists() and (p / "study").exists():
            return p.resolve()
    return None
def _find_repo():
    for cand in (pathlib.Path.cwd().parent, pathlib.Path.cwd(), pathlib.Path("tinybamdp_repo")):
        if _project(cand):
            return _project(cand)
    target = pathlib.Path("tinybamdp_repo")
    if REPO_URL:
        r = subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(target)], capture_output=True, text=True)
        if r.returncode == 0 and _project(target):
            return _project(target)
        print("could not clone", REPO_URL, "- falling back to tinybamdp.zip:", r.stderr.strip()[:200])
    zp = pathlib.Path("tinybamdp.zip")
    if not zp.exists():
        try:
            from google.colab import files
            zp = pathlib.Path(next(iter(files.upload())))
        except ImportError:
            raise FileNotFoundError("Open this notebook from the repository, put tinybamdp.zip next to it, or set REPO_URL.")
    zipfile.ZipFile(zp).extractall(target)
    if _project(target) is None:
        raise FileNotFoundError(f"{zp} does not contain the tinybamdp package and the study folder.")
    return _project(target)
REPO = _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("repository:", REPO)

### Shared imports and UI helpers

In [ ]:
import importlib, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, HTML, clear_output
import tinybamdp
for m in list(sys.modules):
    if m.startswith("tinybamdp"): importlib.reload(sys.modules[m])
from tinybamdp import (bandit_dp, bamdp, beliefs, fitting, models, pomdp, task,
                       get_model, simulate, sample_task, log_likelihood)
import study
for m in list(sys.modules):
    if m.startswith("study"): importlib.reload(sys.modules[m])
from study import analysis, plots
plots.style()
fitting.MP_CONTEXT = "spawn"     # forking a running Jupyter kernel can deadlock; spawn is safe
C = plots.COLORS
AUTORUN = os.environ.get("TB_AUTORUN") == "1"     # used for headless testing only

_CSS = '''<style>
.tb-scroll{max-height:300px;overflow:auto;border:1px solid #d9d8d3;border-radius:6px;background:#fff}
.tb-scroll table{border-collapse:collapse;font-size:12.5px;color:#0b0b0b;width:100%}
.tb-scroll th{position:sticky;top:0;background:#fdeef5;color:#0b0b0b;padding:5px 9px;text-align:right}
.tb-scroll td{padding:4px 9px;border-bottom:1px solid #eeede9;text-align:right}
.tb-scroll tr:hover td{background:#fdf5f9}
.tb-card{background:#fdf0f6;color:#0b0b0b;border-left:4px solid #DC267F;padding:8px 12px;border-radius:4px;margin:6px 0;font-size:13px}
.tb-warn{border-left-color:#FE6100;background:#fff4ec}
</style>'''
def table(df, height=300, index=True, fmt="{:.3f}"):
    html = df.to_html(index=index, float_format=lambda x: fmt.format(x), border=0)
    return HTML(_CSS + f'<div class="tb-scroll" style="max-height:{height}px">{html}</div>')
def card(text, warn=False):
    return HTML(_CSS + f'<div class="tb-card {"tb-warn" if warn else ""}">{text}</div>')
def outbox(h=None):
    lay = W.Layout(border="1px solid #e4e3df", padding="4px", overflow="auto")
    if h: lay.max_height = f"{h}px"
    return W.Output(layout=lay)
def button(desc, style="primary", icon=""):
    return W.Button(description=desc, button_style=style, icon=icon, layout=W.Layout(width="auto"))
SL = dict(continuous_update=False, style={"description_width": "110px"}, layout=W.Layout(width="330px"))
print("tinybamdp", tinybamdp.__version__)

## Quick start (about a minute)

### Observation
Simulate one subject who plans Bayes-optimally, fit the myopic (KG) and optimal (OPT) models plus a model-free learner (RW),
and ask which one the data favour. The rest of the notebook explains each piece; the companion notebook
`kg_vs_optimal_identifiability.ipynb` asks how reliably this comparison works across task designs.

### Mathematical formulation
The subject plays $G=20$ games of $T=15$ trials on $K=4$ arms, so $N=GT=300$ choices. Each model is fitted by maximum likelihood,
and models are compared with the Bayesian information criterion

$$\mathrm{BIC}=2\,\mathrm{NLL}+k\ln N,$$

where NLL is the minimised negative log-likelihood and $k$ the number of free parameters. The **margin** is the BIC difference
between the best and second-best model. A margin below 0.1 counts as a tie; a margin above 2 is conventionally read as positive
evidence and above 6 as strong evidence (Kass & Raftery, 1995).

In [ ]:
rng = np.random.default_rng(0)
subject = simulate(get_model("OPT"), {"m": 0.5, "n": 4.0, "tau": 10.0},   # prior mean, prior strength, choice precision
                   sample_task(20, 4, rng), 15, rng)                       # 20 games x 15 trials on 4 arms
fits = {m: fitting.fit(m, subject, n_starts=3, rng=rng) for m in ("RW", "KG", "OPT")}
winner, margin = fitting.classify(fits, list(fits), tie_eps=0.1)
display(table(pd.DataFrame({m: {"NLL": f.nll, "BIC": f.bic, **f.params} for m, f in fits.items()}).T, height=160))
display(card(f"Best model by BIC: <b>{winner}</b> · margin to runner-up: <b>{margin:.1f}</b> BIC "
             f"({'strong' if margin > 6 else 'positive' if margin > 2 else 'weak'} evidence)"))

### Analysis
With `seed 0` the generating model (OPT) wins with a margin of about 8 BIC over RW, which is strong evidence. KG is third.
The KG fit compensates with a larger prior strength $n$ and a higher choice precision $\tau$ than the OPT fit: the two models
can trade $n$ against $\tau$, which § 7 and the companion study return to. One subject is one random draw: change the seed and
the margin changes. How often the comparison picks the right model is the question of the companion notebook.

## 1 · Unit tests

### Observation
The tests (core and study) check closed-form posteriors, a hand-computed two-trial bandit value, that the vectorised bandit solver equals the generic BAMDP solver, that the optimal policy dominates KG and greedy, likelihood normalisation, MLE recovery, and known tiger-problem values.

In [ ]:
run_tests = button("Run unit tests", "success", "check")
tests_out = outbox(320)
def _run_tests(_=None):
    import subprocess
    with tests_out:
        clear_output()
        r = subprocess.run([sys.executable, "-m", "pytest", "-q", str(REPO / "tests")], capture_output=True, text=True, cwd=REPO)
        print(r.stdout[-3000:] or r.stderr[-3000:])
run_tests.on_click(_run_tests)
display(run_tests, tests_out)
_run_tests()

### Analysis
All 52 tests should pass (`52 passed`). A failure means the installed library does not match the code this notebook was written for; fix that before reading the results below.

## 2 · Beliefs: Beta–Bernoulli conjugacy

### Observation
An arm pays a reward with an unknown rate $\theta$. Press **Observe win** or **Observe loss** to feed outcomes to a Bayesian
learner and watch its belief about $\theta$ change. The sliders set the prior.

### Mathematical formulation
With prior $\theta\sim\mathrm{Beta}(a,b)$, after $s$ wins and $f$ losses the posterior is

$$\theta\mid s,f\sim\mathrm{Beta}(a+s,\;b+f),\qquad
P(\text{win})=\frac{a+s}{a+b+s+f},\qquad
\mathrm{Var}(\theta)=\frac{(a+s)(b+f)}{(a+b+s+f)^2\,(a+b+s+f+1)}.$$

The pair $(s,f)$ is therefore a sufficient statistic: the **hyperstate** that BAMDP planning operates on. The prior is
parameterised by its mean $m=a/(a+b)$ and strength $n=a+b$, so $a=mn$ and $b=(1-m)n$; the fitted models use the same
parameterisation.

In [ ]:
from scipy.stats import beta as _beta
m_s = W.FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05, description="prior mean m", **SL)
n_s = W.FloatLogSlider(value=4, base=10, min=-0.7, max=1.5, step=0.05, description="prior strength n", **SL)
bw, bl, br = button("Observe win", "success", "plus"), button("Observe loss", "danger", "minus"), button("Reset", "", "undo")
obs = {"s": 0, "f": 0, "hist": []}
b_out = outbox()
def draw_belief(*_):
    a0, b0 = m_s.value * n_s.value, (1 - m_s.value) * n_s.value
    a, b = a0 + obs["s"], b0 + obs["f"]
    with b_out:
        clear_output(wait=True)
        x = np.linspace(1e-3, 1 - 1e-3, 400)
        fig, ax = plt.subplots(figsize=(6, 2.6))
        ax.plot(x, _beta.pdf(x, a0, b0), color=plots.MUTED, ls="--", lw=1.5, label="prior")
        ax.plot(x, _beta.pdf(x, a, b), color=C["RW"], label="posterior")
        ax.fill_between(x, _beta.pdf(x, a, b), color=C["RW"], alpha=0.12)
        ax.set_xlabel("reward rate θ"), ax.set_ylabel("density"), ax.legend()
        plt.show()
        bb = beliefs.BetaBelief(a, b)
        display(card(f"<b>Beta({a:.2f}, {b:.2f})</b> · wins {obs['s']} · losses {obs['f']} · "
                     f"mean {bb.mean:.3f} · sd {np.sqrt(bb.var):.3f} · history: {''.join(obs['hist'][-40:]) or '–'}"))
def _obs(o):
    obs["s" if o else "f"] += 1; obs["hist"].append("W" if o else "L"); draw_belief()
bw.on_click(lambda _: _obs(1)); bl.on_click(lambda _: _obs(0))
def _reset(_): obs.update(s=0, f=0, hist=[]); draw_belief()
br.on_click(_reset)
for w in (m_s, n_s): w.observe(draw_belief, "value")
display(W.HBox([m_s, n_s]), W.HBox([bw, bl, br]), b_out); draw_belief()

### Analysis
With the defaults ($m=0.5$, $n=4$) the prior is $\mathrm{Beta}(2,2)$: mean 0.5, standard deviation 0.224. One win moves it to
$\mathrm{Beta}(3,2)$ with mean 0.6. Every observation adds 1 to $a+b$, so a larger prior strength $n$ makes each outcome move the
mean less. This is why $n$ behaves like inertia in the fitted models, and why it can be traded against the choice precision
$\tau$ when fitting (§ 7).

## 3 · Exact Bayes-adaptive planning for a bandit

### Observation
Pick a hyperstate (wins and losses on each of two arms), a horizon and a prior, and compare what the Bayes-optimal planner (OPT)
and the myopic Knowledge-Gradient policy (KG) would choose there.

### Mathematical formulation
For a $K$-armed bandit played for $T$ trials, the BAMDP state is the count vector $h=(s_1,f_1,\dots,s_K,f_K)$. Its Bellman
equation (Bellman, 1957), with $n$ observations so far, is

$$Q_n(h,k)=p_k(h)\,[1+V_{n+1}(h+e^{s}_k)]+(1-p_k(h))\,V_{n+1}(h+e^{f}_k),\qquad V_n(h)=\max_k Q_n(h,k),\qquad V_T\equiv 0,$$

with $p_k(h)$ the posterior predictive of arm $k$ (§ 2) and $e^{s}_k$, $e^{f}_k$ the unit vectors that add a win or a loss to arm
$k$ (Duff, 2002). Reachable hyperstates with $n$ observations are the integer $2K$-vectors summing to $n$, so the problem is a
finite layered graph with $\binom{T+2K}{2K}$ states in total (10,626 for $K=2,T=20$). `bandit_dp` builds the graph once per
$(K,T)$ and solves it with one vectorised pass per layer. Because all arms share one prior, states that differ only by reordering
the arms have the same value, so the graph stores one sorted (*canonical*) state per class: about 24 times fewer states with 4 arms.
Solving takes about a millisecond, fast enough to run inside every likelihood evaluation during fitting.

**Knowledge Gradient** (Frazier et al., 2008; Ryzhov et al., 2012; Zhang & Yu, 2013) assumes that after one more observation no further learning
happens. With posterior means $\mu_j$ and $\mu'_j$ the means after one more observation of arm $k$,

$$\mathrm{KG}_k=\mathbb{E}\big[\max_j\mu'_j\big]-\max_j\mu_j,\qquad D_k=\mu_k+(T-t-1)\,\mathrm{KG}_k,$$

on trial $t$ (counted from 0). Both policies choose by softmax, $P(k)=e^{\tau v_k}/\sum_j e^{\tau v_j}$, with $v=Q$ for OPT and
$v=D$ for KG. Because only value *differences* enter the softmax, the panel plots each policy's preference for arm 1 over arm 2.

In [ ]:
T_s = W.IntSlider(value=10, min=2, max=30, description="horizon T", **SL)
pm_s = W.FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05, description="prior mean m", **SL)
pn_s = W.FloatLogSlider(value=4, base=10, min=-0.5, max=1.3, step=0.05, description="prior strength n", **SL)
cnt = [W.IntSlider(value=v, min=0, max=10, description=d, **SL) for v, d in
       zip([2, 1, 0, 0], ["arm 1 wins", "arm 1 losses", "arm 2 wins", "arm 2 losses"])]
p_out = outbox()
def draw_planner(*_):
    T = T_s.value
    s1, f1, s2, f2 = (c.value for c in cnt)
    n_obs = s1 + f1 + s2 + f2
    with p_out:
        clear_output(wait=True)
        if n_obs >= T:
            display(card(f"This hyperstate has {n_obs} observations, so the game (T={T}) is over. Reduce the counts.", warn=True)); return
        a, b = pm_s.value * pn_s.value, (1 - pm_s.value) * pn_s.value
        t0 = time.perf_counter(); Q, V = bandit_dp.solve(2, T, a, b); dt = 1e3 * (time.perf_counter() - t0)
        g = bandit_dp.bandit_graph(2, T)
        h = np.array([[s1, f1, s2, f2]])
        q = g.lookup(n_obs, h, Q[n_obs])[0]
        S, F = h[:, 0::2].astype(float), h[:, 1::2].astype(float)
        kg = models.KnowledgeGradient.kg_values(a, b, S, F, T - n_obs - 1)[0]
        mu = (a + S[0]) / (a + b + S[0] + F[0])
        df = pd.DataFrame({"posterior mean": mu, "KG value D_k": kg, "Bayes-optimal Q": q,
                           "info bonus (Q − greedy value)": q - (mu + (T - n_obs - 1) * mu.max())},
                          index=["arm 1", "arm 2"])
        # the two value scales differ (KG: one-trial value, OPT: whole remaining game),
        # so plot each policy's preference for arm 1 over arm 2 and the implied choice probability
        fig, axes = plt.subplots(1, 2, figsize=(7.5, 2.2))
        names, cols = ["KG", "OPT"], ["KG value D_k", "Bayes-optimal Q"]
        adv = [df[c].iloc[0] - df[c].iloc[1] for c in cols]
        axes[0].barh(names, adv, color=[C[n] for n in names], height=0.5)
        axes[0].axvline(0, color=plots.MUTED, lw=1); axes[0].set_title("preference: value(arm 1) − value(arm 2)")
        for tau, ls in [(5, "-"), (20, "--")]:
            axes[1].barh([f"{n}  τ={tau}" for n in names], [1 / (1 + np.exp(-tau * a)) for a in adv],
                         color=[C[n] for n in names], height=0.5, alpha=1 if tau == 5 else 0.55)
        axes[1].axvline(0.5, color=plots.MUTED, lw=1); axes[1].set_xlim(0, 1); axes[1].set_title("P(choose arm 1) under softmax")
        for ax in axes: ax.invert_yaxis()
        plt.tight_layout(); plt.show()
        display(card(f"KG picks <b>{df['KG value D_k'].idxmax()}</b> · Bayes-optimal picks <b>{df['Bayes-optimal Q'].idxmax()}</b>",
                     warn=df['KG value D_k'].idxmax() != df['Bayes-optimal Q'].idxmax()))
        display(table(df, index=True))
        display(card(f"Trial {n_obs + 1} of {T} · {T - n_obs - 1} trials remain after this one · "
                     f"graph: <b>{g.n_states:,}</b> canonical hyperstates ({g.n_full_states:,} before arm symmetry) · solve time <b>{dt:.2f} ms</b> · "
                     f"Bayes-optimal value of the whole game from scratch: <b>{V[0][0]:.3f}</b>"))
for w in [T_s, pm_s, pn_s, *cnt]: w.observe(draw_planner, "value")
display(W.HBox([W.VBox([T_s, pm_s, pn_s]), W.VBox(cnt)]), p_out); draw_planner()

### Analysis
The default hyperstate is trial 4 of a 10-trial game with prior $\mathrm{Beta}(2,2)$: arm 1 has 2 wins and 1 loss (posterior
mean 0.571), arm 2 is untried (mean 0.5). **The two policies disagree.** KG gives the untried arm the higher value
(0.586 vs 0.571), because one more look at it could change which arm is best. OPT prefers arm 1 (Q = 4.140 vs 4.113), because
over the six remaining trials the known good arm is worth more. The choice probabilities differ only modestly: P(arm 1) is
0.48 (KG) vs 0.53 (OPT) at $\tau=5$, and 0.43 vs 0.63 at $\tau=20$. Separating the models therefore needs many choices from
states like this one.

Two checks to try: (i) set the counts so that at most two choices remain (e.g. 8 observations with $T=10$); KG and OPT then always
make the same choice, because KG's one-step lookahead is exact there. (ii) The status line reports 511 canonical states for
$K=2,T=10$ (1,001 before the arm-symmetry reduction) and a Bayes-optimal value of 5.660 for the whole game from scratch.

## 4 · Value of information on a generic BAMDP

### Observation
A **known safe arm** pays with probability `p_safe`; an **unknown arm** has a uniform prior (mean 0.5). Compare a planner that
values information (Bayes-adaptive) with one that does not (certainty-equivalent, CE), by exact value and by simulated returns.

### Mathematical formulation
`TabularBAMDP` handles any tabular MDP whose transitions are unknown (Duff, 2002). Each $(s,a)$ pair, or each *tied group* of
pairs, has a Dirichlet belief with counts $c$ over next states, giving the predictive $\hat P(s'\mid s,a,c)$. The Bayes-adaptive
value with $h$ steps to go is

$$Q_h(s,c,a)=\sum_{s'}\hat P(s'\mid s,a,c)\,\big[R(s,a,s')+\max_{a'}Q_{h-1}(s',\,c+\mathbb{1}_{s,a\to s'},\,a')\big],\qquad Q_0\equiv 0,$$

so planning looks ahead over how the counts, and hence the beliefs, will change. The CE planner instead solves the ordinary MDP
with the posterior-mean transitions $\bar P=\hat P(\cdot\mid c)$ held fixed, so it gives no value to learning. Whenever
`p_safe` > 0.5 (the unknown arm's prior mean) CE always picks the safe arm and never learns; at `p_safe` < 0.5 it picks the unknown
arm for exploitation, not exploration.

In [ ]:
def safe_vs_unknown(p_safe=0.55, conc=1e6):
    # states: 0 = last trial lost, 1 = last trial won; actions: 0 = safe, 1 = unknown
    R = np.zeros((2, 2, 2)); R[:, :, 1] = 1
    prior = np.zeros((2, 2, 2))
    prior[:, 0] = [conc * (1 - p_safe), conc * p_safe]      # ≈ known
    prior[:, 1] = [1.0, 1.0]                                 # uniform belief
    tie = np.array([[0, 1], [0, 1]])
    return bamdp.TabularBAMDP(R, prior, tie)
H_s = W.IntSlider(value=10, min=1, max=16, description="horizon H", **SL)
ps_s = W.FloatSlider(value=0.55, min=0.3, max=0.8, step=0.05, description="p_safe", **SL)
N_s = W.IntSlider(value=400, min=100, max=2000, step=100, description="rollouts", **SL)
go = button("Run rollouts", "primary", "play")
v_out = outbox()
def run_voi(_=None):
    with v_out:
        clear_output(wait=True)
        H, ps = H_s.value, ps_s.value; M = safe_vs_unknown(ps); rng = np.random.default_rng(0)
        v_bayes = M.value(0, M.empty_counts(), H)
        ret = {"Bayes-adaptive": [], "certainty-equiv.": []}
        for _ in range(N_s.value):
            th = rng.uniform(); P = np.zeros((2, 2, 2)); P[:, 0] = [1 - ps, ps]; P[:, 1] = [1 - th, th]
            seed = int(rng.integers(1 << 30))
            ret["Bayes-adaptive"].append(M.rollout(P, 0, H, np.random.default_rng(seed), "bayes"))
            ret["certainty-equiv."].append(M.rollout(P, 0, H, np.random.default_rng(seed), "ce"))
        df = pd.DataFrame({k: [np.mean(v), np.std(v) / np.sqrt(len(v))] for k, v in ret.items()},
                          index=["mean return", "s.e."]).T
        fig, ax = plt.subplots(figsize=(5, 2.2))
        ax.barh(df.index, df["mean return"], xerr=1.96 * df["s.e."], color=[C["OPT"], C["WSLS"]], height=0.5)
        ax.set_xlabel(f"return over {H} trials (95% CI)"); plt.tight_layout(); plt.show()
        display(table(df))
        display(card(f"Exact Bayes-adaptive value <b>{v_bayes:.3f}</b> vs CE value <b>{max(ps, 0.5) * H:.3f}</b> "
                     f"(always safe) · hyperstates explored: {len(M._memo):,}"))
go.on_click(run_voi)
display(W.HBox([H_s, ps_s]), W.HBox([N_s, go]), v_out); run_voi()

### Analysis
With the defaults (horizon 10, `p_safe` = 0.55, 400 rollouts) the exact Bayes-adaptive value is 6.091 against 5.5 for CE (always
safe). The simulated means agree: about 6.26 (s.e. 0.11) vs 5.57 (s.e. 0.08). The gap of roughly 0.6 rewards over 10 trials is
the value of information: the Bayes-adaptive planner tries the unknown arm because what it learns can pay off over the remaining
trials, an exploration bonus that the BAMDP produces without any added term. With horizon 1 there is nothing left to learn for, and
the two planners coincide. The memoised recursion visits 1,321 hyperstates here; the count grows quickly with the horizon.

## 5 · POMDP: the tiger problem

### Observation
A tiger is behind the left or right door (Kaelbling et al., 1998). Listening costs 1 and reports the correct side with probability
`acc`; opening the tiger's door costs 100, the other door pays 10, and opening resets the problem. The plot shows the optimal value
and action for every belief; **Simulate an episode** plays the optimal policy once.

### Mathematical formulation
The belief $b(s)=P(s)$ is updated by the discrete Bayes filter after action $a$ and observation $o$:

$$b'(s')=\frac{O(o\mid s',a)\sum_s T(s'\mid s,a)\,b(s)}{P(o\mid b,a)}.$$

The finite-horizon value function

$$V_h(b)=\max_a\Big[\sum_s b(s)R(a,s)+\sum_o P(o\mid b,a)\,V_{h-1}(b')\Big],\qquad V_0\equiv0,$$

is piecewise-linear and convex in $b$, so it is stored as a set of alpha-vectors, $V_h(b)=\max_i\alpha_i\cdot b$. The code backs it up
at the points of a dense belief grid (point-based value iteration; Pineau et al., 2003). With 2 states and 401 grid points this
matches the exact solution at the grid points.

In [ ]:
th_s = W.IntSlider(value=4, min=1, max=12, description="horizon", **SL)
acc_s = W.FloatSlider(value=0.85, min=0.55, max=0.99, step=0.01, description="listen accuracy", **SL)
ep = button("Simulate an episode", "primary", "play")
t_out, e_out = outbox(), outbox(260)
ACOL = {0: C["RW"], 1: C["WSLS"], 2: C["KG"]}
_tiger = {}
def draw_tiger(*_):
    P = pomdp.tiger(acc_s.value); grid = pomdp.belief_grid_2(401)
    layers = P.solve(th_s.value, grid); _tiger.update(P=P, layers=layers)
    with t_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(6.5, 2.8))
        b = grid[:, 0]
        for h, ls in [(th_s.value, "-"), (1, ":")]:
            vals = np.array([P.evaluate(layers[h - 1], g) for g in grid])
            if ls == "-":
                for a in range(3):
                    msk = vals[:, 1] == a
                    ax.scatter(b[msk], vals[msk, 0], s=6, color=ACOL[a], label=P.action_names[a])
            else:
                ax.plot(b, vals[:, 0], ls=ls, lw=1, color=plots.MUTED, label="horizon 1")
        ax.set_xlabel("belief P(tiger-left)"), ax.set_ylabel("value")
        ax.legend(fontsize=8, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.22), markerscale=3)
        ax.set_title(f"V_{th_s.value}(b): colour = optimal action"); plt.show()
        display(card(f"{len(layers[-1][0])} alpha-vectors at horizon {th_s.value}"))
def run_episode(_=None):
    P, layers = _tiger["P"], _tiger["layers"]
    rng = np.random.default_rng()
    s0 = int(rng.integers(2))
    tr = P.simulate(layers, np.array([.5, .5]), s0, rng)
    df = pd.DataFrame([{"step": i + 1, "P(tiger-left)": b[0], "action": P.action_names[a],
                        "observation": P.obs_names[o] if a == 0 else "(reset)", "reward": r}
                       for i, (b, a, o, r) in enumerate(tr)]).set_index("step")
    with e_out:
        clear_output(wait=True)
        display(card(f"Tiger was behind the <b>{P.state_names[s0]}</b> door · total reward <b>{df.reward.sum():.0f}</b>"))
        display(table(df, height=220, fmt="{:.2f}"))
for w in (th_s, acc_s): w.observe(draw_tiger, "value")
ep.on_click(run_episode)
display(W.HBox([th_s, acc_s]), t_out, ep, e_out); draw_tiger(); run_episode()

### Analysis
With horizon 4 and `acc` = 0.85 the value function has 5 alpha-vectors. The optimal policy is to listen unless the belief is
extreme: it opens the right door once $P(\text{tiger-left})\ge 0.935$, opens the left door once it is $\le 0.0675$ (to the
resolution of the grid), and listens in between. At $b=0.5$ the value of the 4-step problem is 2.42. With horizon 1 (dotted line)
opening a door has an expected value of $-45$ at $b=0.5$, so even then listening ($-1$) is best. The simulated episode is random, so
its trace changes each time you press the button.

## 6 · Five cognitive models of bandit choice

### Observation
Simulate a subject from any of the five models and look at its learning curve, its behavioural signatures and its raw choices;
**Compare all models** simulates all five on the same 200 games at mid-range parameters.

### Mathematical formulation
Each subject plays $G$ games of $T$ trials on $K$ arms. In every game the arm rates are redrawn from an environment prior,
$\theta_k\sim\mathrm{Beta}(2,2)$ by default; there is no external dataset. All models share one agent interface (`init`,
`probs`, `update`) that is used both to **simulate** and to compute the **likelihood**, so the two code paths cannot drift apart.

| Model | Learning | Choice | Free parameters |
|---|---|---|---|
| **RW** | delta rule $Q_k\leftarrow Q_k+\eta(r-Q_k)$, $Q_k=0.5$ at the start of each game (Rescorla & Wagner, 1972) | softmax on $Q$ | $\eta,\ \tau$ |
| **WSLS** | none | win-stay / lose-shift with lapse $\epsilon$ | $\epsilon$ |
| **KG** | Bayesian, prior $\mathrm{Beta}(mn,(1-m)n)$ | softmax on Knowledge-Gradient values $D_k$ (§ 3) | $m,\ n,\ \tau$ |
| **KG-DBM** | forgetful Bayes: rates persist with probability $\gamma$, else are redrawn from the prior (Dynamic Belief Model; Yu & Cohen, 2009) | softmax on KG values | $m,\ n,\ \gamma,\ \tau$ |
| **OPT** | Bayesian, prior $\mathrm{Beta}(mn,(1-m)n)$ | softmax on exact BAMDP Q-values (§ 3) | $m,\ n,\ \tau$ |

Choice rules: softmax $P(k)=e^{\tau v_k}/\sum_j e^{\tau v_j}$. WSLS chooses uniformly on the first trial; afterwards, after a win it
repeats the last arm with probability $1-\epsilon+\epsilon/K$, and after a loss it chooses each other arm with probability
$(1-\epsilon)/(K-1)+\epsilon/K$ (Robbins, 1952; Wilson & Collins, 2019). KG-DBM keeps each arm's belief $b$ on a grid of 101 rates;
after the Bayesian update it relaxes towards the prior, $b\leftarrow\gamma\,b+(1-\gamma)\,p_0$ (Zhang & Yu, 2013).
Deterministic policies have zero-probability choices, which makes the log-likelihood $-\infty$, so every model has a noise
term ($\tau$ or $\epsilon$).

In [ ]:
mdl = W.ToggleButtons(options=list(models.MODELS), value="OPT", description="model")
sT = W.IntSlider(value=10, min=2, max=30, description="trials / game T", **SL)
sG = W.IntSlider(value=24, min=4, max=200, description="games G", **SL)
sK = W.IntSlider(value=2, min=2, max=4, description="arms K", **SL)
ea_s = W.FloatSlider(value=2.0, min=0.5, max=5, step=0.5, description="env prior a", **SL)
eb_s = W.FloatSlider(value=2.0, min=0.5, max=5, step=0.5, description="env prior b", **SL)
seed_w = W.IntText(value=1, description="seed", layout=W.Layout(width="160px"))
pbox = W.VBox()
def _param_widgets(*_):
    ws = []
    for p in get_model(mdl.value).params:
        lo, hi = p.sample_range(); mid = (lo + hi) / 2 if p.kind == "unit" else np.sqrt(lo * hi)
        if p.kind == "unit":
            ws.append(W.FloatSlider(value=round(mid, 2), min=p.lo, max=p.hi, step=0.01, description=p.name, **SL))
        else:
            ws.append(W.FloatLogSlider(value=mid, base=10, min=np.log10(p.lo), max=np.log10(p.hi), step=0.02, description=p.name, **SL))
    pbox.children = ws
mdl.observe(_param_widgets, "value"); _param_widgets()
sim_b, cmp_b = button("Simulate subject", "primary", "play"), button("Compare all models", "info", "bar-chart")
s_out, d_out = outbox(), outbox(260)
CURRENT = {}
def _learning_curve(d):
    better = (d.thetas.argmax(1)[:, None] == d.choices)
    return better.mean(0)
def run_sim(_=None):
    m = get_model(mdl.value); prm = {w.description: w.value for w in pbox.children}
    rng = np.random.default_rng(seed_w.value)
    if mdl.value == "OPT" and sK.value == 4 and sT.value > 15:
        with s_out:
            clear_output(); display(card("OPT with 4 arms is limited to T ≤ 15 here (the exact solver grows as C(T+8, 8)).", warn=True))
        return
    d = simulate(m, prm, sample_task(sG.value, sK.value, rng, ea_s.value, eb_s.value), sT.value, rng)
    CURRENT.update(data=d, model=mdl.value, params=prm)
    with s_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(6, 2.5))
        ax.plot(np.arange(1, d.T + 1), _learning_curve(d), "o-", color=C[mdl.value])
        ax.axhline(0.5, color=plots.MUTED, ls=":", lw=1)
        ax.set_ylim(0, 1); ax.set_xlabel("trial"), ax.set_ylabel("P(choose better arm)")
        ax.set_title(f"{mdl.value} · " + ", ".join(f"{k}={v:.2f}" for k, v in prm.items())); plt.show()
        sig = analysis.behavioural_signatures(d)
        display(table(pd.DataFrame(sig, index=[mdl.value]), height=90))
    with d_out:
        clear_output(wait=True)
        rows = [{"game": g + 1, **{f"θ arm{k + 1}": d.thetas[g, k] for k in range(d.K)},
                 "choices": "".join(str(c + 1) for c in d.choices[g]),
                 "rewards": "".join(str(r) for r in d.rewards[g]), "total": d.rewards[g].sum()} for g in range(d.G)]
        display(card("Raw data (one row per game; choices are arm numbers, rewards 0/1). This subject is used in § 7."))
        display(table(pd.DataFrame(rows).set_index("game"), height=220, fmt="{:.2f}"))
def compare(_=None):
    K = sK.value if not (sK.value == 4 and sT.value > 15) else 2
    rng = np.random.default_rng(seed_w.value); th = sample_task(200, K, rng, ea_s.value, eb_s.value); rows = {}
    for name in models.MODELS:
        m = get_model(name)
        prm = {p.name: (np.mean(p.sample_range()) if p.kind == "unit" else np.sqrt(np.prod(p.sample_range())))
               for p in m.params}
        rows[name] = analysis.behavioural_signatures(simulate(m, prm, th, sT.value, rng))
    df = pd.DataFrame(rows).T
    with s_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 2.8)); w = 0.16; x = np.arange(df.shape[1]); k = len(df.index)
        for i, name in enumerate(df.index):
            ax.bar(x + (i - (k - 1) / 2) * w, df.loc[name], w * 0.9, color=C[name], label=name)
        ax.set_xticks(x, df.columns); ax.set_ylim(0, 1.05); ax.legend(ncol=5, fontsize=8)
        ax.set_title(f"Behavioural signatures at mid-range parameters (K={K}, T={sT.value}, 200 games)"); plt.show()
        display(table(df, height=180))
sim_b.on_click(run_sim); cmp_b.on_click(compare)
display(mdl, W.HBox([W.VBox([sT, sG, sK, ea_s, eb_s, seed_w]), pbox]), W.HBox([sim_b, cmp_b]), s_out, d_out); run_sim()

### Analysis
**Compare all models** at the defaults (2 arms, 10 trials, 200 games, seed 1, mid-range parameters) gives these signatures:

| Model | win-stay | lose-shift | choose best mean | choose least tried |
|---|---|---|---|---|
| RW | 0.88 | 0.58 | 0.81 | 0.33 |
| WSLS | 0.85 | 0.86 | 0.69 | 0.46 |
| KG | 0.81 | 0.55 | 0.81 | 0.35 |
| KG-DBM | 0.74 | 0.65 | 0.72 | 0.42 |
| OPT | 0.81 | 0.56 | 0.80 | 0.33 |

WSLS stands out by its high lose-shift rate, and KG-DBM by choosing the less-tried arm more often (its forgetting keeps beliefs
uncertain). RW, KG and OPT are almost indistinguishable on these summary statistics, so separating them needs the trial-by-trial
likelihood (§ 7) rather than signatures. The default single subject (OPT, 24 games) has a learning curve that moves between
about 0.4 and 0.7 from trial to trial: with only 24 games per trial position, individual points are noisy.

## 7 · Fitting all models to one subject

### Observation
Fit all five models to the subject simulated in § 6 and rank them. Fitting takes a few seconds (longer with 4 arms), so it runs
only when you press the button.

### Mathematical formulation
Parameters are optimised in an unconstrained space: $z=\mathrm{logit}(x)$ for parameters in $(0,1)$ ($\eta,\epsilon,m,\gamma$) and
$z=\log x$ for positive ones ($n,\tau$), within box bounds. The objective is the negative log-likelihood of the observed choices
$c_{g,t}$,

$$\mathrm{NLL}(z)=-\sum_{g=1}^{G}\sum_{t=1}^{T}\log P_z(c_{g,t}\mid\text{history}),$$

minimised with L-BFGS-B from several random starts; the best start is kept. Models are compared with
$\mathrm{BIC}=2\,\mathrm{NLL}+k\ln N$. Models within 0.1 BIC of the best are marked as ties, and $\Delta$BIC > 2 / > 6 is
conventionally read as positive / strong evidence (Kass & Raftery, 1995).

In [ ]:
rs_w = W.IntSlider(value=4, min=1, max=10, description="restarts", **SL)
fit_b = button("Fit all models", "primary", "cogs")
f_out = outbox()
def run_fit(_=None):
    d = CURRENT["data"]; rng = np.random.default_rng(0)
    with f_out:
        clear_output(wait=True)
        t0 = time.perf_counter()
        fits = {n: fitting.fit(n, d, rs_w.value, rng) for n in models.MODELS}
        winner, margin = fitting.classify(fits, list(fits), tie_eps=0.1)
        df = pd.DataFrame([{"model": n, "k": get_model(n).n_params, "NLL": f.nll, "BIC": f.bic,
                            "ΔBIC": f.bic - min(x.bic for x in fits.values()), "converged": f.converged,
                            "parameters": ", ".join(f"{k}={v:.2f}" for k, v in f.params.items())}
                           for n, f in fits.items()]).set_index("model").sort_values("BIC")
        strength = "a tie" if winner == fitting.TIE else ("strong" if margin > 6 else "positive" if margin > 2 else "weak")
        display(card(f"Generating model: <b>{CURRENT['model']}</b> ({', '.join(f'{k}={v:.2f}' for k, v in CURRENT['params'].items())}) · "
                     f"best by BIC: <b>{df.index[0] if winner != fitting.TIE else 'tie'}</b> · margin to runner-up {margin:.2f} "
                     f"(evidence: {strength}) · {d.n_trials} choices · {time.perf_counter() - t0:.1f} s",
                     warn=winner != CURRENT["model"]))
        display(table(df, height=200, fmt="{:.2f}"))
fit_b.on_click(run_fit)
display(W.HBox([rs_w, fit_b]), f_out)
if AUTORUN: run_fit()

### Analysis
For the default subject (OPT with $m=0.5$, $n\approx3.16$, $\tau\approx9.49$; 24 games × 10 trials = 240 choices; 4 restarts)
the ranking is RW (BIC 246.5), OPT (248.3), KG (250.1), KG-DBM (253.5), WSLS (304.6). **The generating model loses to RW by
1.8 BIC**, which is weak evidence either way. This is the pattern the companion study quantifies: on short 2-arm games a
delta-rule win is weak evidence that a delta-rule learner generated the data. WSLS is clearly rejected, and the OPT fit recovers
parameters close to the truth ($m=0.46$, $n=2.94$, $\tau=9.05$). Try 4 arms and 15 trials in § 6 to see the margins grow.

### Where to go next
The identifiability study is in **`kg_vs_optimal_identifiability.ipynb`**.

### References
- Bellman, R. (1957). *Dynamic programming*. Princeton University Press.
- Duff, M. O. (2002). *Optimal learning: Computational procedures for Bayes-adaptive Markov decision processes* [Doctoral dissertation, University of Massachusetts Amherst].
- Frazier, P. I., Powell, W. B., & Dayanik, S. (2008). A knowledge-gradient policy for sequential information collection. *SIAM Journal on Control and Optimization, 47*(5), 2410–2439. https://doi.org/10.1137/070693424
- Kaelbling, L. P., Littman, M. L., & Cassandra, A. R. (1998). Planning and acting in partially observable stochastic domains. *Artificial Intelligence, 101*(1–2), 99–134. https://doi.org/10.1016/S0004-3702(98)00023-X
- Kass, R. E., & Raftery, A. E. (1995). Bayes factors. *Journal of the American Statistical Association, 90*(430), 773–795. https://doi.org/10.1080/01621459.1995.10476572
- Pineau, J., Gordon, G., & Thrun, S. (2003). Point-based value iteration: An anytime algorithm for POMDPs. In *Proceedings of the Eighteenth International Joint Conference on Artificial Intelligence* (pp. 1025–1031). IJCAI. https://ijcai.org/proceedings/2003/146
- Rescorla, R. A., & Wagner, A. R. (1972). A theory of Pavlovian conditioning: Variations in the effectiveness of reinforcement and nonreinforcement. In A. H. Black & W. F. Prokasy (Eds.), *Classical conditioning II: Current research and theory* (pp. 64–99). Appleton-Century-Crofts.
- Robbins, H. (1952). Some aspects of the sequential design of experiments. *Bulletin of the American Mathematical Society, 58*(5), 527–535. https://doi.org/10.1090/S0002-9904-1952-09620-8
- Ryzhov, I. O., Powell, W. B., & Frazier, P. I. (2012). The knowledge gradient algorithm for a general class of online learning problems. *Operations Research, 60*(1), 180–195. https://doi.org/10.1287/opre.1110.0999
- Wilson, R. C., & Collins, A. G. E. (2019). Ten simple rules for the computational modeling of behavioral data. *eLife, 8*, Article e49547. https://doi.org/10.7554/eLife.49547
- Yu, A. J., & Cohen, J. D. (2009). Sequential effects: Superstition or rational behavior? In *Advances in Neural Information Processing Systems* (Vol. 21, pp. 1873–1880). Curran Associates.
- Zhang, S., & Yu, A. J. (2013). Forgetful Bayes and myopic planning: Human learning and decision-making in a bandit setting. In *Advances in Neural Information Processing Systems* (Vol. 26, pp. 2607–2615). Curran Associates.